# Main analysis

Use the data from ver 250810.

## Package Loading

In [ ]:
install.packages("pacman")

In [ ]:
pacman::p_load(
  rio, 
  here, 
  skimr, 
  tidyverse, 
  gtsummary, 
  rstatix, 
  janitor, 
  scales, 
  flextable 
  )

## Data Preparation

In [ ]:
setwd("/home/jupyter/")
df_all <- import("all_250810.csv")

head(df_all)

In [ ]:
nrow(df_all)

In [ ]:
df_all <- df_all[-c(1, 2), ]
head(df_all)

In [ ]:
colnames(df_all) <- as.character(unlist(df_all[1, ]))
df_all <- df_all[-1, ]
rownames(df_all) <- NULL

head(df_all)
nrow(df_all)

In [ ]:
cut_row <- which(grepl("kmm", df_all$id_baseline_all))[1]
df_clean <- df_all[1:(cut_row-1), ]

tail(df_clean)

In [ ]:
dim(df_clean)

In [ ]:
colnames(df_clean)

In [ ]:
names(df_clean)[is.na(names(df_clean))] <- "X1"

In [ ]:
table(df_clean$site, useNA="ifany")

In [ ]:
table(df_clean$area, useNA="ifany")

In [ ]:
df_clean <- df_clean %>%
  mutate(
    sitec = case_when(
      site == 1                         ~ 1,
      site == 2                         ~ 2,
      site == 3                         ~ 3,
      site == 4 & area == 1             ~ 4,
      site == 4 & area == 4             ~ 5,
      site == 5                         ~ 6,
      site == 6                         ~ 7,
      site == 7                         ~ 8,
      site == 8                         ~ 9,
      site == 9                         ~ 10,
      site == 10                        ~ 11,
      site == 11                        ~ 12,
      site == 12                        ~ 13,
      site == 13                        ~ 14,
      TRUE                              ~ NA_real_
    )
  )

In [ ]:
table(df_clean$site, df_clean$sitec, useNA="ifany")

In [ ]:
head(df_clean)

## Data integration

### 1. Geographic information

In [ ]:
df_adi_wi <- import("adi_wi_250810.csv")

head(df_adi_wi)

In [ ]:
nrow(df_adi_wi)

In [ ]:
df_adi_wi <- df_adi_wi[-c(1, 2), ]

head(df_adi_wi)

In [ ]:
colnames(df_adi_wi) <- as.character(unlist(df_adi_wi[1, ]))
df_adi_wi <- df_adi_wi[-1, ]
rownames(df_adi_wi) <- NULL

head(df_adi_wi)

In [ ]:
cut_row <- which(grepl("kmm", df_adi_wi$id_baseline_all))[1]
df_adi_wi_clean <- df_adi_wi[1:(cut_row-1), ]

tail(df_adi_wi_clean)

In [ ]:
dim(df_adi_wi_clean)

In [ ]:
colnames(df_adi_wi_clean)

In [ ]:
df_check <- df_clean |>
  transmute(
    id_baseline_all,
    id_gwas_clean = as.character(id_gwas)
  ) |>
  left_join(
    df_adi_wi_clean |>
      transmute(
        id_baseline_all,
        id_gwas_adi = as.character(id_gwas)
      ) %>%
      distinct(id_baseline_all, .keep_all = TRUE),
    by = "id_baseline_all"
  )

df_check |>
  summarise(
    n_total = n(),
    n_both_available = sum(!is.na(id_gwas_clean) & !is.na(id_gwas_adi)),
    n_mismatch = sum(
      !is.na(id_gwas_clean) &
        !is.na(id_gwas_adi) &
        id_gwas_clean != id_gwas_adi
    )
  )

In [ ]:
df_adi_wi_clean_unique <- df_adi_wi_clean |>
  select(-id_gwas, -id_japonica, -id_asa, -site) |>
  distinct(id_baseline_all, .keep_all = TRUE)

df_clean_m <- df_clean |>
  left_join(df_adi_wi_clean_unique, by = "id_baseline_all")

head(df_clean_m)

In [ ]:
df_clean2 <- df_clean_m |>
  filter(!is.na(id_gwas) & id_gwas != "")

head(df_clean2)

In [ ]:
dim(df_clean2)

In [ ]:
names(df_clean2)

### 2. PRS information

In [ ]:
df_prs <- import("ssd/PRS_result/PGS002379_total_PRS.csv")
head(df_prs)

In [ ]:
nrow(df_prs)

In [ ]:
df_merged <- df_clean2 |>
  inner_join(df_prs, by = c("id_gwas" = "IID"))
head(df_merged)

In [ ]:
nrow(df_merged)

In [ ]:
sum(df_merged$id_gwas %in% df_prs$IID)

In [ ]:
sum(!df_merged$id_gwas %in% df_prs$IID)

In [ ]:
colnames(df_merged)

### 3. Site name

In [ ]:
site_table <- read.csv("site_table2.csv")
site_table$site_num <- as.character(site_table$site_num)
head(site_table)

In [ ]:
df_merged$sitec <- as.character(df_merged$sitec)
class(df_merged$site)

In [ ]:
df_merged <- df_merged %>%
  left_join(site_table, by = c("sitec" = "site_num"))

df_merged <- df_merged %>%
  mutate(site_name = factor(site_name, levels = site_table$site_name))

In [ ]:
head(df_merged)

## Definition of Type 2 Diabetes

In [ ]:
df_merged <- df_merged |>
  mutate(
    PRS_quintile = ntile(PRS_z, 5),
    
    pdm = factor(
      pdm,
      levels = c(1, 2, 3),
      labels = c("なし", "あり", "以前あり")
    )
  )

table(df_merged$PRS_quintile, df_merged$pdm)

In [ ]:
plot_df <- df_merged %>%
  mutate(
    dm_bin = ifelse(
      pdm %in% c("あり", "以前あり"),
      1,
      0
    )
  ) %>%
  group_by(PRS_quintile) %>%
  summarise(
    dm_rate = mean(dm_bin, na.rm = TRUE) * 100
  )

plot_df

In [ ]:
ggplot(plot_df,
       aes(x = factor(PRS_quintile),
           y = dm_rate)) +
  geom_col() +
  labs(
    x = "PRS quintile",
    y = "Diabetes prevalence (%)"
  )

In [ ]:
df_merged <- df_merged %>%
  mutate(
    dm_bin = ifelse(
      pdm %in% c("あり", "以前あり"),
      1,
      0
    )
  )

table(df_merged$dm_bin)

### Logistic regression of PRS between Diabetes mellitus

In [ ]:
fit <- glm(dm_bin ~ PRS_z,
           data = df_merged,
           family = binomial())

summary(fit)

PRS associated with Diabetes.

## Add Walkability Index(JPWI) variables

In [ ]:
df_merged <- df_merged |>
  mutate(
    jpwi = parse_number(as.character(jpwi))
  )#parse_number 文字列の中から数値だけ取り出して数値型に変換する

class(df_merged$jpwi)

In [ ]:
df_merged |>
  group_by(site_name) |>
  summarise(
    n = sum(!is.na(jpwi)),
    mean_jpwi = mean(jpwi, na.rm = TRUE),
    sd_jpwi = sd(jpwi, na.rm = TRUE),
    median_jpwi = median(jpwi, na.rm = TRUE),
    min_jpwi = min(jpwi, na.rm = TRUE),
    max_jpwi = max(jpwi, na.rm = TRUE),
    .groups = "drop"
  )# .groups = "drop"でgroup_byのグループ化解除

### Histogram of JPWI

In [ ]:
ggplot(df_merged, aes(x = jpwi)) +
  geom_histogram(bins = 50) +
  labs(
    x = "Walkability Index",
    y = "Count",
    title = "Distribution of Walkability Index"
  ) +
  theme_classic()

### Box plot of JPWI by site

In [ ]:
df_merged |>
  mutate(site_num = as.numeric(site),
         site = factor(site_num)) |>
  ggplot(aes(x = site_name, y = jpwi, fill = site)) +
  geom_boxplot() +
  scale_fill_brewer(palette = "Set3") +
  labs(
    x = "Site",
    y = "Walkability index"
  ) +
  theme_bw()+
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme(legend.position = "none")

### Standarized JPWI

In [ ]:
df_merged$jpwi_z <- as.numeric(scale(df_merged$jpwi))

summary(df_merged$jpwi_z)
sd(df_merged$jpwi_z, na.rm = TRUE)

## Add physical activity variables

### Definition of physical activity(MET-hours/week)

In [ ]:
vars <- c('physicalwork', 'walking', 'standing', 'sitting', 
          'exer_freq_light', 'exert_time_light', 'exer_freq_mod', 'exer_time_mod', 'exer_freq_heavy', 'exert_time_heavy')

df_merged <- df_merged %>%
  mutate(across(all_of(vars), ~ ifelse(. %in% c(0, 9), NA, .)))

vars <- c("exer_freq_light","exert_time_light",
          "exer_freq_mod","exer_time_mod",
          "exer_freq_heavy","exert_time_heavy")


counts_list <- lapply(df_merged[vars], function(x) {
  sort(table(x, useNA = "always"), decreasing = TRUE)
})

counts_list

In [ ]:
value_map_freq <- c(
  "1" = 0,
  "2" = 0.5,
  "3" = 1.5,
  "4" = 3.5,
  "5" = 6
)

columns_to_map_freq <- c(
  "exer_freq_light",
  "exer_freq_mod",
  "exer_freq_heavy"
)


df_merged <- df_merged |>
mutate(
    across(
        all_of(columns_to_map_freq),
        ~ value_map_freq[as.character(.)],
        .names = "{.col}r"
    )
)

table(df_merged$exer_freq_light, df_merged$exer_freq_lightr, useNA = "ifany")
table(df_merged$exer_freq_mod, df_merged$exer_freq_modr, useNA = "ifany")
table(df_merged$exer_freq_heavy, df_merged$exer_freq_heavyr, useNA = "ifany")

In [ ]:
value_map_time <- c(
  "1" = 0.25,
  "2" = 0.75,
  "3" = 1.5,
  "4" = 2.5,
  "5" = 3.5,
  "6" = 4.0
)

columns_to_map_time <- c(
  "exert_time_light",
  "exer_time_mod",
  "exert_time_heavy"
)

df_merged <- df_merged |>
mutate(
    across(
        all_of(columns_to_map_time),
        ~ value_map_time[as.character(.)],
        .names = "{.col}r"
    )
)

table(df_merged$exert_time_light, df_merged$exert_time_lightr, useNA = "ifany")
table(df_merged$exer_time_mod, df_merged$exer_time_modr, useNA = "ifany")
table(df_merged$exert_time_heavy, df_merged$exert_time_heavyr, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(
    weak = 3.4 * exer_freq_lightr * exert_time_lightr,
    weak = if_else(exer_freq_light == 1, 0, weak),
   
    moderate = 7 * exer_freq_modr * exer_time_modr,
    moderate = if_else(exer_freq_mod == 1, 0, moderate),

    intense = 10 * exer_freq_heavyr * exert_time_heavyr,
    intense = if_else(exer_freq_heavy == 1, 0, intense)
  )

df_merged <- df_merged %>%
mutate(
    methw = weak + moderate +intense
    )

class(df_merged$methw)
summary(df_merged$methw)
sum(!is.na(df_merged$methw)) 

In [ ]:
sum(df_merged$methw == 0, na.rm = TRUE) 

mean(df_merged$methw == 0, na.rm = TRUE) * 100

### box Plot and violin plot of physical activity

In [ ]:
df_merged %>%
  group_by(site_name) %>%
  summarise(
    n = n(),
    mean = mean(methw, na.rm = TRUE),
    sd = sd(methw, na.rm = TRUE),
    median = median(methw, na.rm = TRUE),
    Q1 = quantile(methw, 0.25, na.rm = TRUE),
    Q3 = quantile(methw, 0.75, na.rm = TRUE),
    min = min(methw, na.rm = TRUE),
    max = max(methw, na.rm = TRUE),
    zero_n = sum(methw == 0, na.rm = TRUE),
    zero_pct = mean(methw == 0, na.rm = TRUE) * 100
  )

In [ ]:
ggplot(df_merged, aes(x = methw)) +
  geom_histogram(bins = 50) +
  labs(
    x = "METs/week",
    y = "Count",
    title = "Distribution of METs/week"
  ) +
  theme_minimal()

df_merged$log_methw <- log1p(df_merged$methw)

ggplot(df_merged, aes(x = log_methw)) +
  geom_histogram(bins = 50) +
  labs(
    x = "log1p(methw)",
    y = "Count",
    title = "Distribution of log1p(methw)"
  ) +
  theme_minimal()

ggplot(df_merged, aes(x = site_name, y = methw)) +
  geom_violin(fill = "grey90", color = "grey40", trim = FALSE) +
  geom_jitter(
    width = 0.15,
    alpha = 0.15,
    size = 0.8
  ) +
  labs(
    x = "Site",
    y = "MET-hours/week",
    title = "Distribution of physical activity by site"
  ) +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  )

### Log-transform physical activity

In [ ]:
df_merged$log_methw <- log1p(df_merged$methw)

summary(df_merged$log_methw)

In [ ]:
table(df_merged$site, is.na(df_merged$methw))

## Creation of covariates

### drinking status

In [ ]:
table(df_merged$dr_yn, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(
    alcohol = case_when(
      dr_yn %in% c(1, 2, 3) ~ dr_yn,
      dr_yn %in% c(0, 9) ~ NA_character_
    )
  )

table(df_merged$alcohol, useNA = "ifany")

### smoking status

In [ ]:
table(df_merged$sm_yn, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(smoking = case_when(
         sm_yn %in% c(1, 2, 3) ~ sm_yn,
         sm_yn %in% c(0, 9) ~ NA_character_
    )
  )

table(df_merged$smoking, useNA = "ifany")

### Age

In [ ]:
df_merged$age <- as.numeric(df_merged$age)
class(df_merged$age)

### BMI

In [ ]:
df_merged$bmi <- as.numeric(df_merged$bmi)
class(df_merged$bmi)

df_merged$bmi[df_merged$bmi == 99999] <- NA

summary(df_merged$bmi)

## Creation of complete-Case Dataset

In [ ]:
df_complete <- df_merged %>%
  filter(
    complete.cases(
      PRS_z,
      sitec,
      age,
      sex,
      bmi,
      alcohol,
      smoking,
      jpwi,    
      jpwi_z,
      methw,
      log_methw
    )
  )

In [ ]:
nrow(df_complete)

In [ ]:
demo <- df_complete %>%
  summarise(
    N = n(),
    Age_min = min(age, na.rm = TRUE),
    Age_mean = mean(age, na.rm = TRUE),
    Age_sd = sd(age, na.rm = TRUE),
    Age_max = max(age, na.rm = TRUE),
    Male_n = sum(sex == 1, na.rm = TRUE),
    Female_n = sum(sex == 2, na.rm = TRUE)
  )

demo

In [ ]:
df_complete$sitec <- factor(df_complete$sitec)
df_complete$sitec <- relevel(df_complete$sitec, ref = "11")

# Main analysis

## Association of JPWI with Diabetes

#### adjusted for age, sex and site

In [ ]:
m_dm_jpwi1 <- glm(dm_bin ~ jpwi_z + age + sex + sitec, 
               data = df_complete, 
               family = binomial())

summary(m_dm_jpwi1)

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
m_dm_jpwi2 <- glm(dm_bin ~ jpwi_z + age + sex + sitec + PRS_z + smoking + alcohol, 
               data = df_complete, 
               family = binomial())

summary(m_dm_jpwi2)

In [ ]:
coef_tab <- summary(m_dm_jpwi2)$coefficients

result <- data.frame(
  Variable = rownames(coef_tab),
  OR  = exp(coef_tab[, "Estimate"]),
  LCL = exp(coef_tab[, "Estimate"] -
              1.96 * coef_tab[, "Std. Error"]),
  UCL = exp(coef_tab[, "Estimate"] +
              1.96 * coef_tab[, "Std. Error"]),
  P   = coef_tab[, "Pr(>|z|)"]
)

result

### Interaction analysis of JPWI and site

#### adjusted for age and sex

In [ ]:
m_dm_jpwi3 <- glm(dm_bin ~ age + sex + sitec*jpwi_z, 
               data = df_complete, 
               family = binomial())

summary(m_dm_jpwi3)

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
m_dm_jpwi4 <- glm(dm_bin ~ age + sex + sitec*jpwi_z + PRS_z + smoking + alcohol, 
               data = df_complete, 
               family = binomial())

summary(m_dm_jpwi4)

### Likelihood Ratio Test

#### adjusted for age and sex

In [ ]:
fit0 <- glm(
  dm_bin ~ age + sex + sitec + jpwi_z,
  family = binomial(),
  data = df_complete
)
fit1 <- glm(
  dm_bin ~ age + sex + sitec * jpwi_z,
  family = binomial(),
  data = df_complete
)

anova(fit0, fit1, test = "LRT")

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
fit0 <- glm(
  dm_bin ~ age + sex + sitec + jpwi_z + PRS_z + smoking + alcohol,
  family = binomial(),
  data = df_complete
)
fit1 <- glm(
  dm_bin ~ age + sex + sitec * jpwi_z + PRS_z + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

anova(fit0, fit1, test = "LRT")

## Association of phsical activity and Diabetes

#### adjusted for age, sex and site

In [ ]:
m_dm_logmethw1 <- glm(dm_bin ~ log_methw + age + sex + sitec,
                          data = df_complete,
                          family = binomial())

summary(m_dm_logmethw1)

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
m_dm_logmethw2 <- glm(dm_bin ~ log_methw + age + sex + sitec + PRS_z + smoking + alcohol,
                          data = df_complete,
                          family = binomial())

summary(m_dm_logmethw2)

In [ ]:
coef_tab <- summary(m_dm_logmethw2)$coefficients

result <- data.frame(
  Variable = rownames(coef_tab),
  OR  = exp(coef_tab[, "Estimate"]),
  LCL = exp(coef_tab[, "Estimate"] -
              1.96 * coef_tab[, "Std. Error"]),
  UCL = exp(coef_tab[, "Estimate"] +
              1.96 * coef_tab[, "Std. Error"]),
  P   = coef_tab[, "Pr(>|z|)"]
)

result

### Interaction analysis of physical activity and site

#### adjusted for age and sex

In [ ]:
m_dm_logmethw3 <- glm(dm_bin ~ log_methw * sitec + age + sex,
                          data = df_complete,
                          family = binomial())

summary(m_dm_logmethw3)

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
m_dm_logmethw4 <- glm(dm_bin ~ log_methw * sitec + age + sex + PRS_z + smoking + alcohol,
                          data = df_complete,
                          family = binomial())

summary(m_dm_logmethw4)

### Likelihood Ratio Test

#### adjusted for age and sex

In [ ]:
model_methw_no_int <- glm(
  dm_bin ~ log_methw + sitec + age + sex,
  family = binomial(),
  data = df_complete
)

model_methw_int <- glm(
  dm_bin ~ log_methw * sitec + age + sex,
  family = binomial(),
  data = df_complete
)

anova(model_methw_no_int, model_methw_int, test = "Chisq")

#### more adjusted for PRS_z, smoking status and drinking status

In [ ]:
model_methw_no_int <- glm(
  dm_bin ~ log_methw + sitec + age + sex + PRS_z + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

model_methw_int <- glm(
  dm_bin ~ log_methw * sitec + age + sex + PRS_z + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

anova(model_methw_no_int, model_methw_int, test = "Chisq")

## Association of PRS with Daibetes

#### adjusted for age, sex amd site

In [ ]:
m_PRS_dm1 <- glm(dm_bin ~ PRS_z + sitec + age + sex,
                  data = df_complete,
                  family = binomial())

summary(m_PRS_dm1)

#### more asjusted for BMI, smoking status and drinking status

In [ ]:
m_PRS_dm2 <- glm(dm_bin ~ PRS_z + sitec + age + sex + bmi + alcohol + smoking,
                  data = df_complete,
                  family = binomial())

summary(m_PRS_dm2)

In [ ]:
coef_tab <- summary(m_PRS_dm2)$coefficients

result <- data.frame(
  Variable = rownames(coef_tab),
  OR  = exp(coef_tab[, "Estimate"]),
  LCL = exp(coef_tab[, "Estimate"] -
              1.96 * coef_tab[, "Std. Error"]),
  UCL = exp(coef_tab[, "Estimate"] +
              1.96 * coef_tab[, "Std. Error"]),
  P   = coef_tab[, "Pr(>|z|)"]
)

result

### Interaction analysis of PRS and site

#### adjusted for age and sex

In [ ]:
m_PRS_site1 <- glm(dm_bin ~ PRS_z * sitec + age + sex,
                  data = df_complete,
                  family = binomial())

summary(m_PRS_site1)

#### more adjusted for bmi, smoking, alcohol

In [ ]:
m_PRS_site2 <- glm(dm_bin ~ PRS_z * sitec + age + sex + bmi + smoking + alcohol,
                  data = df_complete,
                  family = binomial())

summary(m_PRS_site2)

### Likelihood Ratio Test

#### adjusted for age and sex

In [ ]:
model_prs_no_int <- glm(
  dm_bin ~ PRS_z + sitec + age + sex,
  family = binomial(),
  data = df_complete
)

model_prs_int <- glm(
  dm_bin ~ PRS_z * sitec + age + sex,
  family = binomial(),
  data = df_complete
)

anova(model_prs_no_int, model_prs_int, test = "Chisq")

#### more adjusted for bmi, smoking, alcohol

In [ ]:
model_prs_no_int <- glm(
  dm_bin ~ PRS_z + sitec + age + sex + bmi + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

model_prs_int <- glm(
  dm_bin ~ PRS_z * sitec + age + sex + bmi + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

anova(model_prs_no_int, model_prs_int, test = "Chisq")